In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import librosa

In [5]:
from pathlib import Path
import pandas as pd

# ---- paths ----
PROJECT_ROOT = Path("..").resolve()
DATASETS_ROOT = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/datasets/Androids")

INTERVIEW_HC_DIR = DATASETS_ROOT / "Interview-Task" / "audio" / "HC"
INTERVIEW_PT_DIR = DATASETS_ROOT / "Interview-Task" / "audio" / "PT"
READING_HC_DIR = DATASETS_ROOT / "Reading-Task" / "HC"
READING_PT_DIR = DATASETS_ROOT / "Reading-Task" / "PT"

METADATA_DIR = DATASETS_ROOT

# ---- load raw csvs ----
control = pd.read_csv(METADATA_DIR / "bdi-control.csv")
depression = pd.read_csv(METADATA_DIR / "bdi-depression.csv")
folds_raw = pd.read_csv(METADATA_DIR / "fold-lists.csv", header=None)

In [6]:
# ---- clean label files ----
control.columns = [c.strip().lower() for c in control.columns]
depression.columns = [c.strip().lower() for c in depression.columns]

control["depressed"] = 0
depression["depressed"] = 1

labels = pd.concat([control, depression], ignore_index=True)
labels = labels.rename(columns={"bdi": "bdi_score"})
labels["file"] = labels["file"].astype(str).str.strip()
labels["file_stem"] = labels["file"].str.replace(".wav", "", regex=False).str.strip()
labels["file_lower"] = labels["file"].str.lower()

In [7]:

# ---- list audio files from all four folders ----
audio_records = []

audio_sources = [
    (INTERVIEW_HC_DIR, "interview", "HC"),
    (INTERVIEW_PT_DIR, "interview", "PT"),
    (READING_HC_DIR, "read", "HC"),
    (READING_PT_DIR, "read", "PT"),
]

for folder, speech_type, subgroup in audio_sources:
    wavs = list(folder.glob("*.wav"))
    for p in wavs:
        audio_records.append({
            "file_path": str(p.resolve()),
            "file": p.name.strip(),
            "file_stem": p.stem.strip(),
            "file_lower": p.name.strip().lower(),
            "speech_type_from_path": speech_type,
            "subgroup_from_path": subgroup,
        })

audio_df = pd.DataFrame(audio_records)

if len(audio_df) == 0:
    raise ValueError("No .wav files found in the four Androids audio folders.")

In [8]:
def extract_fold_section(df, col_start, col_end, speech_type):
    section = df.iloc[2:, col_start:col_end].copy()
    section.columns = df.iloc[1, col_start:col_end]
    section = section.reset_index(drop=True)

    long_df = section.melt(var_name="fold", value_name="file_stem")
    long_df["speech_type"] = speech_type

    long_df["file_stem"] = (
        long_df["file_stem"]
        .astype(str)
        .str.strip()
        .str.replace("'", "", regex=False)
        .str.replace('"', "", regex=False)
    )

    long_df["fold"] = long_df["fold"].astype(str).str.strip()

    long_df = long_df[long_df["file_stem"].notna()]
    long_df = long_df[long_df["file_stem"] != ""]
    long_df = long_df[~long_df["file_stem"].str.lower().eq("nan")]

    return long_df


read_folds = extract_fold_section(folds_raw, 0, 5, "read")
interview_folds = extract_fold_section(folds_raw, 7, 12, "interview")
folds_tidy = pd.concat([read_folds, interview_folds], ignore_index=True)

audio_df["file_stem"] = audio_df["file_stem"].astype(str).str.strip()
labels["file_lower"] = labels["file"].astype(str).str.strip().str.lower()


In [9]:

androids_meta = (
    audio_df.merge(
        labels[["file_lower", "bdi_score", "depressed"]],
        on="file_lower",
        how="left"
    )
    .merge(
        folds_tidy[["file_stem", "fold", "speech_type"]],
        on="file_stem",
        how="left"
    )
)

androids_meta["speech_type"] = androids_meta["speech_type"].fillna(androids_meta["speech_type_from_path"])

androids_meta = androids_meta[
    [
        "file_path",
        "file",
        "file_stem",
        "bdi_score",
        "depressed",
        "fold",
        "speech_type",
        "subgroup_from_path",
    ]
].copy()

print("Total audio rows:", len(audio_df))
print("Rows with labels:", androids_meta["bdi_score"].notna().sum())
print("Rows with folds:", androids_meta["fold"].notna().sum())
print("Speech types from merged table:\n", androids_meta["speech_type"].value_counts(dropna=False))

androids_meta = androids_meta.dropna(subset=["bdi_score", "depressed", "fold"]).reset_index(drop=True)

if len(androids_meta) == 0:
    raise ValueError("No rows left after merging audio, labels, and folds. Check filenames and paths.")

androids_meta["depressed"] = androids_meta["depressed"].astype(int)

Total audio rows: 116
Rows with labels: 224
Rows with folds: 226
Speech types from merged table:
 speech_type
interview    116
read         110
Name: count, dtype: int64


In [10]:
print(audio_df["file_stem"].head().tolist())
print(folds_tidy["file_stem"].head().tolist())

audio_stems = set(audio_df["file_stem"])
fold_stems = set(folds_tidy["file_stem"])
overlap = audio_stems.intersection(fold_stems)

print("Overlap:", len(overlap))
print("Example overlap:", list(overlap)[:10])

['01_CF56_1', '02_CM57_2', '03_CF30_3', '04_CF57_3', '05_CF41_3']
['01_CF56_1', '02_CM57_2', '09_CF56_3', '21_CF58_3', '22_CF50_3']
Overlap: 116
Example overlap: ['62_PF56_3', '07_CF50_2', '54_CM48_2', '65_PF41_3', '38_PF31_2', '40_CF59_1', '44_CF37_3', '03_PF66_3', '49_CM54_4', '25_CF59_3']


In [12]:
# ---- feature extraction ----
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    pitch = librosa.yin(y, fmin=50, fmax=300)
    pitch = pitch[np.isfinite(pitch)]
    pitch_mean = np.mean(pitch) if len(pitch) > 0 else np.nan

    energy = np.mean(librosa.feature.rms(y=y))

    return np.concatenate([mfcc_mean, [pitch_mean, energy]])


In [13]:
features = []
valid_rows = []
failed_files = []

for _, row in androids_meta.iterrows():
    try:
        feats = extract_features(row["file_path"])
        if np.any(np.isnan(feats)) or np.any(np.isinf(feats)):
            failed_files.append((row["file"], "NaN or Inf in features"))
            continue
        features.append(feats)
        valid_rows.append(row)
    except Exception as e:
        failed_files.append((row["file"], str(e)))

print("Successful feature rows:", len(features))
print("Failed files:", len(failed_files))
print("First 5 failures:", failed_files[:5])

if len(features) == 0:
    raise ValueError("No features extracted successfully.")

feature_names = [f"mfcc_{i+1}" for i in range(13)] + ["pitch_mean", "energy_mean"]

X = np.vstack(features)
X_df = pd.DataFrame(X, columns=feature_names)

df_model = pd.concat([pd.DataFrame(valid_rows).reset_index(drop=True), X_df], axis=1)

Successful feature rows: 224
Failed files: 0
First 5 failures: []


In [14]:
from sklearn.model_selection import PredefinedSplit

# ---- save processed datasets ----
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df_model.to_csv(processed_dir / "androids_model_dataset_basic.csv", index=False)

# ---- modelling arrays ----
X = df_model[feature_names].values
y = df_model["depressed"].values

fold_map = {"fold1": 0, "fold2": 1, "fold3": 2, "fold4": 3, "fold5": 4}
test_fold = df_model["fold"].astype(str).str.strip().map(fold_map)

if test_fold.isna().any():
    bad = df_model.loc[test_fold.isna(), "fold"].unique()
    raise ValueError(f"Unexpected fold names found: {bad}")

ps = PredefinedSplit(test_fold=test_fold.astype(int).values)

print("Final modelling rows:", len(df_model))
print("Class counts:\n", df_model["depressed"].value_counts())
print("Fold counts:\n", df_model["fold"].value_counts())


Final modelling rows: 224
Class counts:
 depressed
1    122
0    102
Name: count, dtype: int64
Fold counts:
 fold
fold1    46
fold5    45
fold3    45
fold2    45
fold4    43
Name: count, dtype: int64
